# Module 2.5 — Advanced pandas transformations

## Turning administrative records into analysis-ready features

This lesson continues the fictional **Northstar Community Health Service** appointments scenario used across Module 2. The records are entirely fictional and exist only for learning.

You will reshape data, add group-level context to individual records, engineer useful features, create reusable pipelines, and validate the result.

## Learning objectives

By the end of this lesson, you should be able to:

- distinguish aggregation from transformation;
- use `groupby().transform()` without losing appointment rows;
- create features with `assign()`, `map()`, `np.select()`, and `pd.cut()`;
- build ordered patient-history features with `cumcount()` and `shift()`;
- reshape between wide and long forms with `melt()` and `pivot()`;
- flatten hierarchical indexes;
- organise transformations with small functions and `pipe()`;
- validate row counts, identifiers, categories, and calculations.

> Administrative data is often shaped for data entry or reporting. Analysis may require another shape. Every transformation should preserve or intentionally change the meaning of one row.

## Lesson map

1. Rebuild and inspect the fictional dataset.
2. Establish the grain.
3. Compare aggregation and transformation.
4. Engineer row-level and group-relative features.
5. Create ordered history features.
6. Reshape wide and long tables.
7. Build and validate a reusable pipeline.
8. Practise independently.
9. Test your knowledge.

## Setup

Run this cell first. The notebook uses NumPy and pandas only.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 120)
DATA_DIR = Path("data")
OUTPUT_DIR = Path("outputs")
DATA_DIR.mkdir(exist_ok=True)
OUTPUT_DIR.mkdir(exist_ok=True)
print("pandas version:", pd.__version__)

## Fictional administrative dataset

Each row represents one scheduled appointment. A patient can have several appointments, but `appointment_id` must identify exactly one row.

The code creates a deterministic teaching dataset so everyone begins with the same records.

In [ ]:
clinics = ["Central", "North", "South", "East"]
services = ["Primary Care", "Physiotherapy", "Dental", "Mental Health"]
channels = ["Web", "Phone", "Desk", "Referral"]
statuses = ["Completed", "Completed", "Completed", "Completed", "Completed", "No-show", "Cancelled"]
rows = []
start = pd.Timestamp("2026-01-05 08:30")

for i in range(40):
    status = statuses[i % len(statuses)]
    appointment_dt = start + pd.Timedelta(days=i) + pd.Timedelta(minutes=(i % 4) * 45)
    lead_days = 2 + (i % 6)
    completed = status == "Completed"
    rows.append({
        "appointment_id": f"A{1001+i}",
        "patient_id": f"P{1 + (i * 7) % 29:03d}",
        "booking_date": appointment_dt.normalize() - pd.Timedelta(days=lead_days),
        "appointment_datetime": appointment_dt,
        "clinic": clinics[i % 4],
        "service": services[(i * 3) % 4],
        "visit_type": ["Routine", "Initial", "Follow-up", "Urgent"][i % 4],
        "status": status,
        "wait_minutes": 5 + (i * 7) % 38 if completed else np.nan,
        "consultation_minutes": 20 + (i * 5) % 43 if completed else 0,
        "estimated_cost": 48 + (i * 11) % 76,
        "satisfaction_score": 2 + (i % 4) if completed else pd.NA,
        "priority_case": bool(i % 6 == 0),
        "booking_channel": channels[(i * 3) % 4],
    })

appointments = pd.DataFrame(rows)
appointments["satisfaction_score"] = appointments["satisfaction_score"].astype("Int64")
source_path = DATA_DIR / "northstar_appointments.csv"
appointments.to_csv(source_path, index=False)
print(appointments.head().to_string(index=False))

### Inspect types and missingness

Missing satisfaction and waiting values are expected for appointments that were not completed. Missing identifiers would be a structural problem.

In [ ]:
appointments.info()
print(pd.DataFrame({
    "dtype": appointments.dtypes.astype(str),
    "missing_n": appointments.isna().sum(),
    "unique_n": appointments.nunique(dropna=True),
}))

## Establish the grain

State the unit of observation before transforming data:

> One row represents one scheduled appointment.

If a step unexpectedly changes the row count or makes `appointment_id` non-unique, the grain may have changed.

In [ ]:
grain_check = pd.Series({
    "rows": len(appointments),
    "unique_appointment_ids": appointments["appointment_id"].nunique(),
    "duplicate_appointment_ids": appointments["appointment_id"].duplicated().sum(),
    "unique_patients": appointments["patient_id"].nunique(),
})
assert appointments["appointment_id"].is_unique
assert appointments["appointment_id"].notna().all()
print(grain_check.to_string())

## Aggregation versus transformation

An aggregation reduces records into a summary. A transformation returns a value aligned with each input row.

The first table has one row per clinic. The second calculation repeats the appropriate clinic mean beside every appointment.

In [ ]:
clinic_summary = (
    appointments.groupby("clinic", as_index=False)
    .agg(
        scheduled=("appointment_id", "size"),
        completed=("status", lambda s: s.eq("Completed").sum()),
        mean_wait=("wait_minutes", "mean"),
        median_wait=("wait_minutes", "median"),
    )
)
print(clinic_summary.to_string(index=False))

In [ ]:
appointments["clinic_mean_wait"] = (
    appointments.groupby("clinic")["wait_minutes"].transform("mean")
)
appointments["wait_vs_clinic_mean"] = (
    appointments["wait_minutes"] - appointments["clinic_mean_wait"]
)
appointments[["appointment_id", "clinic", "wait_minutes", "clinic_mean_wait", "wait_vs_clinic_mean"]].head(10).to_string(index=False)

### Group-relative rank

A raw wait can be ordinary in one clinic and unusual in another. Percentile rank adds local context. Values nearer 1 represent longer recorded waits within that clinic.

In [ ]:
appointments["clinic_wait_percentile"] = (
    appointments.groupby("clinic")["wait_minutes"].rank(method="average", pct=True)
)
appointments.loc[appointments["wait_minutes"].notna(),
    ["appointment_id", "clinic", "wait_minutes", "clinic_wait_percentile"]
].sort_values(["clinic", "clinic_wait_percentile"]).head(12).to_string(index=False)

## Feature engineering with `assign()`

`assign()` creates columns and returns a new DataFrame. This supports readable workflows without overwriting the source table.

In [ ]:
features = (
    appointments.assign(
        appointment_date=lambda df: df["appointment_datetime"].dt.normalize(),
        appointment_month=lambda df: df["appointment_datetime"].dt.to_period("M").astype(str),
        weekday=lambda df: df["appointment_datetime"].dt.day_name(),
        appointment_hour=lambda df: df["appointment_datetime"].dt.hour,
        lead_days=lambda df: (df["appointment_datetime"].dt.normalize() - df["booking_date"]).dt.days,
        is_completed=lambda df: df["status"].eq("Completed"),
        is_no_show=lambda df: df["status"].eq("No-show"),
        total_recorded_minutes=lambda df: df["wait_minutes"].fillna(0) + df["consultation_minutes"],
    )
)
features[["appointment_id", "appointment_month", "weekday", "lead_days", "is_completed", "total_recorded_minutes"]].head().to_string(index=False)

### Conditional categories

`np.select()` handles ordered, multi-condition rules. The first true condition wins, so place the most specific condition first.

These wait bands are teaching examples, not service-level targets.

In [ ]:
conditions = [
    ~features["is_completed"],
    features["wait_minutes"].le(10),
    features["wait_minutes"].le(20),
]
features["wait_band"] = np.select(
    conditions,
    ["Not applicable", "0–10 minutes", "11–20 minutes"],
    default="Over 20 minutes"
)
features["wait_band"].value_counts()

### Binning and mapping

Use `pd.cut()` for explicit numeric intervals and `map()` for one-to-one label conversion. Always check mapped results: an unknown source value becomes missing.

In [ ]:
features["lead_time_band"] = pd.cut(
    features["lead_days"],
    bins=[-1, 2, 5, np.inf],
    labels=["0–2 days", "3–5 days", "6+ days"]
)
channel_map = {
    "Web": "Self-service", "Phone": "Staff-assisted",
    "Desk": "Staff-assisted", "Referral": "Professional referral"
}
features["channel_group"] = features["booking_channel"].map(channel_map)
assert features["channel_group"].notna().all()
print(features[["booking_channel", "channel_group"]].drop_duplicates())
print(features[["lead_days", "lead_time_band"]].value_counts().sort_index())

## Ordered patient history

Sequence features depend on order. Sort by patient and appointment time before using `cumcount()`, `shift()`, or cumulative calculations.

In [ ]:
features = features.sort_values(
    ["patient_id", "appointment_datetime", "appointment_id"]
).reset_index(drop=True)
features["patient_appointment_number"] = features.groupby("patient_id").cumcount() + 1
features["previous_appointment_datetime"] = (
    features.groupby("patient_id")["appointment_datetime"].shift(1)
)
features["days_since_previous"] = (
    features["appointment_datetime"] - features["previous_appointment_datetime"]
).dt.days
features[["patient_id", "appointment_id", "appointment_datetime", "patient_appointment_number", "days_since_previous"]].head(12).to_string(index=False)

### Prior history without leakage

A prior-no-show feature should use events before the current record. Subtracting the current no-show flag from the cumulative total excludes the present appointment.

In [ ]:
features["prior_no_shows"] = (
    features.groupby("patient_id")["is_no_show"].cumsum()
    - features["is_no_show"].astype(int)
)
features[["patient_id", "appointment_id", "status", "prior_no_shows"]].head(12).to_string(index=False)

## Custom functions and `apply()`

Prefer vectorised operations for simple calculations. Use row-wise `apply()` when genuinely custom multi-column logic is clearer.

The function below creates a transparent operational follow-up label.

In [ ]:
def follow_up_reason(row):
    if row["status"] != "Completed":
        return "Attendance follow-up"
    if row["priority_case"] and row["satisfaction_score"] <= 3:
        return "Priority experience review"
    if row["wait_minutes"] > 30:
        return "Long-wait review"
    return "No immediate follow-up"

features["follow_up_reason"] = features.apply(follow_up_reason, axis=1)
features["follow_up_reason"].value_counts()

## Reshape wide to long

Wide tables store repeated measures in separate columns. Long tables store the measure name and value in separate fields. `melt()` performs this unpivoting operation.

In [ ]:
time_wide = features[[
    "appointment_id", "clinic", "service", "wait_minutes", "consultation_minutes"
]].copy()

time_long = time_wide.melt(
    id_vars=["appointment_id", "clinic", "service"],
    value_vars=["wait_minutes", "consultation_minutes"],
    var_name="time_measure",
    value_name="minutes"
)
assert len(time_long) == len(time_wide) * 2
time_long.head(10).to_string(index=False)

## Reshape long to wide

`pivot()` restores unique appointment–measure pairs to columns. It fails if an index-and-column pair maps to more than one value. If duplicates are valid and require aggregation, use `pivot_table()` with an explicit aggregation rule.

In [ ]:
time_round_trip = (
    time_long.pivot(
        index=["appointment_id", "clinic", "service"],
        columns="time_measure",
        values="minutes"
    ).reset_index().rename_axis(columns=None)
)
original = time_wide.sort_values("appointment_id").reset_index(drop=True)
restored = time_round_trip.sort_values("appointment_id").reset_index(drop=True)
pd.testing.assert_frame_equal(original[restored.columns], restored, check_dtype=False)
print("The wide → long → wide round trip preserved the values.")

## MultiIndex results

Grouping on several fields can create hierarchical row or column indexes. `reset_index()` moves row-index levels back into columns. Column labels can be flattened for export.

In [ ]:
service_month_summary = features.groupby(["appointment_month", "service"]).agg(
    scheduled=("appointment_id", "size"),
    completed=("is_completed", "sum"),
    mean_wait=("wait_minutes", "mean"),
    mean_satisfaction=("satisfaction_score", "mean"),
)
service_month_flat = service_month_summary.reset_index()
service_month_flat.head(8).to_string(index=False)

In [ ]:
multi_agg = features.groupby("clinic").agg({
    "wait_minutes": ["mean", "median", "max"],
    "consultation_minutes": ["mean", "sum"],
})
multi_agg.columns = ["_".join(parts) for parts in multi_agg.columns.to_flat_index()]
print(multi_agg.reset_index().to_string(index=False))

## Reusable pipeline with `pipe()`

Small functions make rules easier to test and reuse. Each function below copies its input or returns a new frame rather than silently mutating another object.

In [ ]:
def add_calendar_features(df):
    result = df.copy()
    result["appointment_month"] = result["appointment_datetime"].dt.to_period("M").astype(str)
    result["weekday"] = result["appointment_datetime"].dt.day_name()
    result["lead_days"] = (result["appointment_datetime"].dt.normalize() - result["booking_date"]).dt.days
    return result

def add_outcome_features(df):
    return df.assign(
        is_completed=df["status"].eq("Completed"),
        is_no_show=df["status"].eq("No-show"),
        is_cancelled=df["status"].eq("Cancelled"),
    )

def add_group_context(df):
    result = df.copy()
    result["clinic_mean_wait"] = result.groupby("clinic")["wait_minutes"].transform("mean")
    result["wait_vs_clinic_mean"] = result["wait_minutes"] - result["clinic_mean_wait"]
    result["service_mean_cost"] = result.groupby("service")["estimated_cost"].transform("mean")
    result["cost_vs_service_mean"] = result["estimated_cost"] - result["service_mean_cost"]
    return result

def validate_analysis_table(df):
    assert df["appointment_id"].is_unique
    assert df["lead_days"].ge(0).all()
    assert df["status"].isin(["Completed", "No-show", "Cancelled"]).all()
    assert df.loc[~df["is_completed"], "wait_minutes"].isna().all()
    return df

analysis_table = (
    appointments.pipe(add_calendar_features)
    .pipe(add_outcome_features)
    .pipe(add_group_context)
    .pipe(validate_analysis_table)
)
analysis_table.head().to_string(index=False)

### Validate and export

A passing assertion confirms only the condition written in that assertion. It does not prove that the underlying business definition is correct.

Export after checking grain, keys, permitted categories, date logic, and feature arithmetic.

In [ ]:
assert len(analysis_table) == len(appointments)
assert analysis_table["appointment_id"].is_unique
recalculated = analysis_table["estimated_cost"] - analysis_table["service_mean_cost"]
pd.testing.assert_series_equal(analysis_table["cost_vs_service_mean"], recalculated, check_names=False)

detail_path = OUTPUT_DIR / "northstar_appointments_analysis_ready.csv"
summary_path = OUTPUT_DIR / "northstar_service_month_summary.csv"
analysis_table.to_csv(detail_path, index=False)
service_month_flat.to_csv(summary_path, index=False)

assert len(pd.read_csv(detail_path)) == len(analysis_table)
assert len(pd.read_csv(summary_path)) == len(service_month_flat)
print("Validated and exported:", detail_path, "and", summary_path)

## Guided practice

Create `service_mean_consultation` using `transform("mean")`, then calculate `consultation_vs_service_mean`.

Before coding, predict whether the number of rows should change.

In [ ]:
guided = analysis_table.copy()
# guided["service_mean_consultation"] = ...
# guided["consultation_vs_service_mean"] = ...
# guided[["appointment_id", "service", "consultation_minutes",
#         "service_mean_consultation", "consultation_vs_service_mean"]].head(8).to_string(index=False)

### Guided-practice solution

The row count does not change because `transform()` returns a result aligned with the input rows.

In [ ]:
guided_solution = analysis_table.copy()
guided_solution["service_mean_consultation"] = (
    guided_solution.groupby("service")["consultation_minutes"].transform("mean")
)
guided_solution["consultation_vs_service_mean"] = (
    guided_solution["consultation_minutes"] - guided_solution["service_mean_consultation"]
)
assert len(guided_solution) == len(analysis_table)
guided_solution[["appointment_id", "service", "consultation_minutes",
                 "service_mean_consultation", "consultation_vs_service_mean"]].head(8).to_string(index=False)

## Independent practice

Using `analysis_table`:

1. Add each clinic’s completion rate to every row with `transform("mean")`.
2. Add a `cost_band`: `Under 60`, `60–89`, or `90+`.
3. Melt `estimated_cost`, `wait_minutes`, and `consultation_minutes` into `operations_long`.
4. Create a clinic-by-month pivot table showing scheduled appointment counts.
5. Validate the expected row count of the long table.
6. Write one careful observation and one question these data cannot answer.

In [ ]:
practice = analysis_table.copy()
# Build your solution here.

### Independent-practice solution

In [ ]:
practice_solution = analysis_table.copy()
practice_solution["clinic_completion_rate"] = (
    practice_solution.groupby("clinic")["is_completed"].transform("mean")
)
practice_solution["cost_band"] = pd.cut(
    practice_solution["estimated_cost"], [-np.inf, 59, 89, np.inf],
    labels=["Under 60", "60–89", "90+"]
)
operations_long = practice_solution.melt(
    id_vars=["appointment_id", "appointment_month", "clinic", "service", "status"],
    value_vars=["estimated_cost", "wait_minutes", "consultation_minutes"],
    var_name="measure", value_name="value"
)
clinic_month_pivot = practice_solution.pivot_table(
    index="clinic", columns="appointment_month", values="appointment_id",
    aggfunc="count", fill_value=0
)
assert len(operations_long) == len(practice_solution) * 3
print(practice_solution[["appointment_id", "clinic", "clinic_completion_rate", "estimated_cost", "cost_band"]].head().to_string(index=False))
print(operations_long.head(9).to_string(index=False))
print(clinic_month_pivot)

**Careful interpretation:** The fictional records show different observed completion rates across clinics during the recorded period. They do not establish why those rates differ, whether the pattern persists, or whether clinic performance caused the difference.

> Transformations reveal patterns but do not create causal evidence.

## Common pitfalls

- Changing the grain accidentally; check row counts and keys after shape-changing operations.
- Building sequence features before sorting.
- Allowing current or future information into a “prior history” feature.
- Ignoring missing values produced by an incomplete mapping.
- Using `pivot()` when index-and-column pairs are duplicated.
- Treating missing values as zero without justification.
- Hiding operational definitions inside opaque code.
- Over-interpreting engineered features.

## Test your knowledge!

Answer before opening the suggested answers.

1. How do `groupby().agg()` and `groupby().transform()` differ?
2. Why sort before using `shift()`?
3. When would you use `melt()`?
4. Why can `pivot()` fail even when all requested columns exist?
5. What does `reset_index()` do after grouping?
6. Why inspect missing results after `map()`?
7. A 40-row table melts three measure columns. How many long rows should result?
8. Why does a passing assertion not prove a feature’s business meaning is correct?

<details>
<summary><strong>Suggested answers</strong></summary>

1. `agg()` reduces groups to summaries; `transform()` returns values aligned with original rows.
2. `shift()` follows current row order, so an unsorted table may identify the wrong previous event.
3. Use `melt()` to turn repeated measure columns into measure-and-value rows.
4. Each index-and-column pair must identify at most one value; duplicates are ambiguous.
5. It moves index levels back into ordinary columns.
6. Unrecognised values become missing and can hide a new or misspelled category.
7. 120 rows.
8. An assertion tests only the programmed condition; the condition may still misrepresent the real process.

</details>

## Stretch challenge

Write `make_patient_history_features(df)` that sorts records, numbers each patient’s appointments, calculates days since the previous appointment, counts prior no-shows without including the current row, and preserves row count and unique appointment IDs.

In [ ]:
def make_patient_history_features(df):
    # Complete the function, then call it with .pipe().
    pass

### Stretch solution

In [ ]:
def make_patient_history_features_solution(df):
    result = df.sort_values(["patient_id", "appointment_datetime", "appointment_id"]).reset_index(drop=True).copy()
    result["patient_appointment_number"] = result.groupby("patient_id").cumcount() + 1
    result["previous_appointment_datetime"] = result.groupby("patient_id")["appointment_datetime"].shift(1)
    result["days_since_previous"] = (result["appointment_datetime"] - result["previous_appointment_datetime"]).dt.days
    result["prior_no_shows"] = result.groupby("patient_id")["is_no_show"].cumsum() - result["is_no_show"].astype(int)
    return result

stretch_solution = analysis_table.pipe(make_patient_history_features_solution)
assert len(stretch_solution) == len(analysis_table)
assert stretch_solution["appointment_id"].is_unique
assert stretch_solution["patient_appointment_number"].ge(1).all()
assert stretch_solution["prior_no_shows"].ge(0).all()
stretch_solution.head().to_string(index=False)

## Lesson close

You have moved from isolated pandas commands to auditable transformation workflows. Keep stating the grain, preserving raw inputs, making rules explicit, choosing operations that match the intended output shape, and validating every shape-changing step.

The exported data remains fictional and must not be used for operational or clinical decisions.